### Aplicação de Regras de Negócio

**Cliente Ativo**
- Cadastro completo (nome, CPF/CNPJ, contato válido)
- Pelo menos um contrato vigente (status ativo)
- Não possui bloqueios administrativos ou pendências críticas

**Cliente Pendente**
- Cadastro incompleto (falta CPF, e-mail ou telefone)
- Contrato em análise ou aguardando documentação
- Pode ter contratos sem valor ou sem data definida
- Status temporário até regularização

**Cliente Bloqueado**
- Restrição por inadimplência, fraude ou decisão administrativa
- Não pode firmar novos contratos
- Mesmo que tenha contratos ativos, eles ficam suspensos ou em cobrança

#### Regras de Negócio — Contratos

**Contrato Ativo**
- Tem valor definido
- Data de início preenchida e dentro da vigência
- Status = Ativo

**Contrato Pendente**
- Falta informação essencial (valor, vencimento, cliente vinculado)
- Status = Pendente

**Contrato Encerrado**
- Data de rescisão preenchida ou vencimento atingido
- Status = Encerrado

In [0]:
# Instalando biblioteca para geração de dados fictícios
%pip install faker

In [0]:
# Importando bibliotecas necessárias
from pyspark.sql import SparkSession
import random
from faker import Faker
from datetime import date, datetime, timedelta
from pyspark.sql.functions import regexp_replace, to_date, col, when, count, max as max_, lit

# Inicializando SparkSession e Faker com locale brasileiro
spark = SparkSession.builder.getOrCreate()
fake = Faker("pt_BR")

In [0]:
# Lendo as tabelas da camada Silver
df_clientes = spark.table("workspace.default.silver_clientes")
df_contratos = spark.table("workspace.default.silver_contratos")
df_agencias = spark.table("workspace.default.bronze_agencias")

print(f"Clientes: {df_clientes.count()} registros")
print(f"Contratos: {df_contratos.count()} registros")
print(f"Agências: {df_agencias.count()} registros")

In [0]:
# Identificando clientes com 3+ parcelas em atraso
clientes_bloqueados = df_contratos.filter(col("parcelas_em_atraso") >= 3) \
    .select("cliente_id") \
    .distinct() \
    .withColumn("flag_parcela_atraso", lit(True))

# Agregando informações de contratos por cliente
contratos_agg = df_contratos.groupBy("cliente_id").agg(
    count("*").alias("total_contratos"),
    max_("data_fim").alias("ultima_data_fim")
)

# Join com clientes e flag de bloqueio
df_clientes_com_contratos = df_clientes \
    .join(contratos_agg, on="cliente_id", how="left") \
    .join(clientes_bloqueados, on="cliente_id", how="left")

# Aplicando regras de classificação de clientes
df_gold_clientes = df_clientes_com_contratos.withColumn(
    "status_cliente",
    when(
        # BLOQUEADO: 3+ parcelas em atraso
        col("flag_parcela_atraso") == True,
        "BLOQUEADO"
    ).when(
        # PENDENTE: cadastro incompleto (falta CPF, email OU telefone)
        (col("cpf_cnpj").isNull()) | 
        (col("email").isNull()) | 
        (col("telefone").isNull()),
        "PENDENTE"
    ).when(
        # ATIVO: cadastro completo E tem pelo menos 1 contrato vigente
        (col("cpf_cnpj").isNotNull()) & 
        (col("email").isNotNull()) & 
        (col("telefone").isNotNull()) &
        (col("total_contratos") > 0) &
        (col("ultima_data_fim") >= date.today()),
        "ATIVO"
    ).otherwise(
        "PENDENTE"  # Cadastro completo mas sem contratos vigentes
    )
)

# Selecionando apenas as colunas relevantes para a camada Gold
df_gold_clientes = df_gold_clientes.select(
    "cliente_id",
    "nome",
    "cpf_cnpj",
    "email",
    "telefone",
    "endereco",
    "agencia_id",
    "data_cadastro",
    "status_cliente",
    "data_particao"
)

df_gold_clientes.groupBy("status_cliente").count().orderBy("status_cliente").show()

In [0]:
# Aplicando regras de classificação de contratos
df_gold_contratos = df_contratos.withColumn(
    "status_contrato",
    when(
        # ENCERRADO: data de fim já passou
        col("data_fim") < date.today(),
        "ENCERRADO"
    ).when(
        # PENDENTE: falta informação essencial (valor, data_inicio, ou cliente)
        (col("valor_total").isNull()) | 
        (col("data_inicio").isNull()) | 
        (col("cliente_id").isNull()),
        "PENDENTE"
    ).when(
        # ATIVO: tem valor, data de início, e ainda está vigente
        (col("valor_total").isNotNull()) & 
        (col("data_inicio").isNotNull()) & 
        (col("data_inicio") <= date.today()) &
        (col("data_fim") >= date.today()),
        "ATIVO"
    ).otherwise(
        "PENDENTE"  # Casos não previstos
    )
)

df_gold_contratos.groupBy("status_contrato").count().orderBy("status_contrato").show()

In [0]:
# Salvando tabela Gold de Clientes
df_gold_clientes.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("data_particao") \
    .saveAsTable("workspace.default.gold_clientes")

# Salvando tabela Gold de Contratos
df_gold_contratos.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("data_particao") \
    .saveAsTable("workspace.default.gold_contratos")